
# 03 · Kết quả — so sánh 8 agents

**Vai trò notebook**: phân tích so sánh đầy đủ cho slide chapter 5 (Results & Discussion).

**Owner**: Person 1
**Deadline**: 2026-05-25
**Slide chapter**: 5 — Results (CRITICAL — headline numbers ở đây)

## Mục tiêu
1. Bảng so sánh đầy đủ 8 agents × (cum return, Sharpe, Sortino, MaxDD, turnover, LLM cost).
2. Multiple figures cho slide: equity curves, ranked bar, Sharpe vs return, drawdown distribution.
3. Multi-agent specific: debate-rounds histogram, parse-failure rate, avg latency.
4. Narrative cho thầy: ai thắng ai, by how much, why, surprises.

## Defense Q&A
- Q: Multi-agent (+50%) vẫn thua buy_and_hold (+103%) — dự án còn ý nghĩa?
- Q: Zero-shot Sharpe 10.52 cao bất thường — explain?
- Q: PPO vs DDPG cùng RL nhưng khác xa — why?
- Q: Sao multi_agent thua zero_shot về Sharpe?

## Single source of truth
Mọi số liệu đọc từ `metrics` = `load_metrics_table()`. **KHÔNG** hardcode số nào.


## Setup


In [ ]:
import sys
from pathlib import Path

# Make `from _shared import ...` work whether you run from notebooks/ or repo root.
_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: master-table — bảng tổng hợp formatted cho slide
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `metrics` DataFrame (đã load ở Setup)
- **WRITE**: `report/figures/03__master_table.md` (markdown table) + render trong notebook
- **CONSTRAINTS**:
  - Columns: agent, category, cum_return (%), Sharpe, Sortino, MaxDD (%), n_steps, LLM cost (USD)
  - Sort by cum_return DESC
  - Format: cum_return + Sharpe ở 2 chữ số thập phân; cost ở format `$X.XX`
  - Highlight row multi_agent (bold) — đó là headline
- **VALIDATE**:
  - 8 rows
  - File exists
  - `metrics.loc['multi_agent', 'cumulative_return']` ≈ 0.5018 (PKG-S snapshot)
- **PATTERN**: `pandas.DataFrame.to_markdown(index=False)` hoặc tabulate
- **DEFENSE Q&A**: "Cho xem bảng kết quả?" → mở file này hoặc render inline


In [ ]:
# TODO-01: master comparison table
print(metrics[['cumulative_return','sharpe','sortino','max_drawdown','n_steps']].sort_values('cumulative_return', ascending=False).to_markdown())



## TODO-02: equity-curves — overlay 8 portfolio curves
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `load_curve(agent)` for each agent
- **WRITE**: `report/figures/03__equity_curves.png`
- **CONSTRAINTS**:
  - X = date, Y = portfolio_value / 1e9 (billion VND)
  - 8 lines: baselines dashed (alpha 0.6), RL solid mid-weight, LLM solid + bold cho multi_agent
  - Color = AGENT_COLORS
  - Horizontal reference line at y=1.0 (initial capital)
  - Annotate final value mỗi line bên phải
  - Title VI: "Đường giá trị danh mục — 248 phiên test 2025-05 → 2026-04"
- **VALIDATE**: 8 lines visible, file exists
- **PATTERN**: `scripts/make_figures.py:fig_portfolio_curves` đã làm 80% việc này
- **DEFENSE Q&A**: "Trực quan hoá kết quả?" → mở figure này


In [ ]:
# TODO-02: equity curves overlay
fig, ax = plt.subplots(figsize=(11, 5.5))
# ...



## TODO-03: ranked-bars — 2 figures (cum_return + Sharpe)
- **OWNER**: Person 1   **DEPENDS**: TODO-01
- **WRITE**: `report/figures/03__cumret_bar.png` + `03__sharpe_bar.png`
- **CONSTRAINTS**:
  - Horizontal bar, sorted ascending (bar dài nhất ở trên cùng để slide đọc top-down)
  - Color = AGENT_COLORS, end-point dot bonus glow
  - Annotate value lên bar
  - Title VI: "Tỷ suất sinh lời tích luỹ", "Sharpe ratio"
- **VALIDATE**: 2 files; cum_return chart có multi_agent ở vị trí thứ 3 (sau buy_and_hold, equal_weight)
- **PATTERN**: `scripts/make_figures.py:fig_cum_return_bar`
- **DEFENSE Q&A**: "Thứ tự xếp hạng?" → mở figure này


In [ ]:
# TODO-03: ranked horizontal bars (2 figures)
pass



## TODO-04: risk-return-scatter — Sharpe vs return scatter
- **OWNER**: Person 1   **DEPENDS**: none
- **WRITE**: `report/figures/03__risk_return_scatter.png`
- **CONSTRAINTS**:
  - X = cum_return, Y = Sharpe
  - Bubble size = n_steps (LLM smoke = nhỏ; full backtest = lớn)
  - Bubble color = AGENT_COLORS
  - Label mỗi điểm với agent name
  - Quadrant lines tại Sharpe=0 (x-axis tô đậm), return=0
  - Title VI: "Risk-adjusted return — Sharpe vs cumulative return"
- **VALIDATE**:
  - 8 points
  - zero_shot ở top-left (high Sharpe, low return — vì smoke 10 sessions không representative)
  - multi_agent middle-upper (high Sharpe + decent return)
- **DEFENSE Q&A**: "Sao zero_shot Sharpe cao mà return thấp?" → smoke 10 sessions không representative; full backtest sẽ regress về median.


In [ ]:
# TODO-04: risk-return scatter
fig, ax = plt.subplots(figsize=(9, 6))
# ...



## TODO-05: drawdown-comparison — max DD bar + worst drawdown periods
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: each agent's portfolio_curve
- **WRITE**: `report/figures/03__drawdown_compare.png`
- **CONSTRAINTS**:
  - Compute drawdown series cho mỗi agent: `dd_t = (running_max - value) / running_max`
  - Plot 8 drawdown lines (signed -, axis từ 0 xuống min)
  - Highlight max DD point mỗi line
  - Title VI: "Drawdown trên 8 chiến lược"
- **VALIDATE**:
  - max DD multi_agent ≈ 16% (per metrics)
  - max DD random > 25% (theo expected)
- **PATTERN**: `frontend/components/DrawdownChart.tsx:computeDrawdownSeries` đã có thuật toán
- **DEFENSE Q&A**: "Risk control của multi_agent thế nào?" → ref max DD < 17% so với random > 26%.


In [ ]:
# TODO-05: drawdown comparison
def compute_dd(curve):
    rm = curve['portfolio_value'].cummax()
    return (rm - curve['portfolio_value']) / rm
# ...



## TODO-06: multi-agent-audit — debate rounds + parse failures + latency
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `results/multi_agent/decisions.jsonl`, `metrics.loc['multi_agent']`
- **WRITE**:
  - `report/figures/03__multi_agent_audit.png` (3-panel: rounds histogram, latency boxplot, cost cumulative)
  - Markdown table: avg_debate_rounds, parse_failure_rate, node_errors_total, avg_latency_s
- **CONSTRAINTS**:
  - All 51 decisions có debate_rounds = 2 (theo PKG-S log)
  - Latency boxplot: median, IQR, outliers
  - Cost cumulative: monotonic increasing, total ≈ $3.21
- **VALIDATE**:
  - 51 decisions
  - parse_failure_rate == 0
  - total cost match metrics.loc['multi_agent', 'llm_cost_usd']
- **DEFENSE Q&A**: "Multi-agent stable không? Có timeout / parse fail?" → 0 timeout, 0 parse fail, 0 node errors qua 51 decisions.


In [ ]:
# TODO-06: multi_agent operational audit
decisions = [json.loads(l) for l in open(RESULTS / 'multi_agent' / 'decisions.jsonl')]
# ...



## TODO-07: narrative-summary — Markdown 1 trang cho slide title
- **OWNER**: Person 1   **DEPENDS**: TODO-01..06
- **WRITE**: `report/figures/03__narrative.md` (1 trang) + render trong notebook
- **CONSTRAINTS**:
  - Headline: "Multi-agent đạt +50.18% (Sharpe 2.19) — xếp thứ 3 sau buy_and_hold (+103%) và equal_weight (+53%)"
  - 4 paragraph:
    1. Big picture — ai thắng (top 3)
    2. RL bracket — PPO +40% vs DDPG +1% (saturated tanh)
    3. LLM bracket — multi_agent > single > zero_shot
    4. Surprises — buy_and_hold vẫn ngon nhất; ý nghĩa: market bullish 2025-2026, alpha agent khó
  - Đọc tất cả số từ `metrics` DataFrame; không hardcode
- **VALIDATE**: file exists, ≥ 500 chars
- **DEFENSE Q&A**: "Tóm tắt 1 câu kết quả?" → paragraph 1 của file này


In [ ]:
# TODO-07: narrative markdown
narrative_path = FIGURES / '03__narrative.md'
# ...



## Defense Q&A — câu trả lời sẵn

> **Q1: Multi-agent thua buy_and_hold — dự án còn ý nghĩa?**
> A: Mục tiêu thesis không phải beat buy_and_hold (passive index) mà là **so sánh tương đối RL vs LLM**. Trong bull market 2025-2026 (VN30 +103%), beating index khó cho mọi active strategy. Quan trọng: multi_agent (Sharpe 2.19) gần ngang buy_and_hold (2.75), gấp đôi PPO Sharpe 1.26 — chứng tỏ LLM agent có alpha *nhưng* không đủ vượt index khi market thuần bull.
> Evidence: TODO-04 risk-return scatter.

> **Q2: Zero-shot Sharpe 10.52 cao bất thường?**
> A: Artifact của smoke run N=10 sessions chỉ, không representative. Annualized Sharpe trên 10 obs over-estimate vì small denominator. Full backtest (như multi_agent N=51) Sharpe ổn định 2.19.
> Evidence: TODO-04 bubble size scaling.

> **Q3: PPO vs DDPG khác xa?**
> A: DDPG saturated tanh (Risk #7) → portfolio stuck overweight HPG → return +1% chỉ. PPO clipped objective + entropy bonus → balanced allocation → +40%.
> Evidence: TODO-02 equity curves (PPO leo dần, DDPG flat) + holdings.

> **Q4: Multi_agent vs zero_shot về Sharpe?**
> A: Cùng smoke vs full constraint. Khi cả 2 full backtest, multi_agent Sharpe 2.19 vs zero_shot dự kiến regress về 1-2. Smoke 10 obs không đáng tin.
